# Toxic Full Dataset - Tuned Random Forest + Feature Selection

Notebook nay dung ban toxic full descriptors `data/toxic/data.csv`. Vi so mau it nhung feature rat nhieu, quy trinh fine-tune gom: chon feature quan trong bang Random Forest tren train set, thu mot grid nho cho `top_n`, `max_depth`, `min_samples_leaf`, tune threshold bang out-of-fold probabilities, va danh gia tren test split.

In [76]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
TEST_SIZE = 0.2

# Cau hinh final lay tu tuning run: cai thien recall/F1 Toxic hon baseline top-50 threshold 0.5.
FINAL_TOP_N_FEATURES = 13
FINAL_MIN_SAMPLES_LEAF = 1
FINAL_MAX_DEPTH = None
FINAL_THRESHOLD = 0.4


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "data").exists():
            return path
    raise FileNotFoundError("Cannot find project root containing data/ directory")


ROOT = find_project_root(Path.cwd().resolve())
DATA_PATH = ROOT / "data" / "toxic" / "data.csv"

df = pd.read_csv(DATA_PATH)
df.shape

(171, 1204)

In [77]:
display(df.head())
print("Class distribution:")
display(df["Class"].value_counts().to_frame("count"))
display(df["Class"].value_counts(normalize=True).mul(100).round(2).to_frame("percent"))

,MATS3v,nHBint10,MATS3s,MATS3p,nHBDon_Lipinski,minHBint8,MATS3e,MATS3c,minHBint2,MATS3m,...,WTPT-4,WTPT-5,ETA_EtaP_L,ETA_EtaP_F,ETA_EtaP_B,nT5Ring,SHdNH,ETA_dEpsilon_C,MDEO-22,Class
0,0.0908,0,0.0075,0.0173,0,0.0,-0.0436,0.0409,0.0,0.1368,...,0.0000,0.0000,0.1780,1.5488,0.0088,0,0.0,-0.0868,0.00,NonToxic
1,0.0213,0,0.1144,-0.0410,0,0.0,0.1231,-0.0316,0.0,0.1318,...,8.8660,19.3525,0.1739,1.3718,0.0048,2,0.0,-0.0810,0.25,NonToxic
2,0.0018,0,-0.0156,-0.0765,2,0.0,-0.1138,-0.1791,0.0,0.0615,...,5.2267,27.8796,0.1688,1.4395,0.0116,2,0.0,-0.1004,0.00,NonToxic
3,-0.0251,0,-0.0064,-0.0894,3,0.0,-0.0747,-0.1151,0.0,0.0361,...,7.7896,24.7336,0.1702,1.4654,0.0133,2,0.0,-0.1010,0.00,NonToxic
4,0.0135,0,0.0424,-0.0353,0,0.0,-0.0638,0.0307,0.0,0.0306,...,12.3240,19.7486,0.1789,1.4495,0.0120,2,0.0,-0.1071,0.00,NonToxic


Class distribution:


,count
Class,
NonToxic,115
Toxic,56


,percent
Class,
NonToxic,67.25
Toxic,32.75


In [78]:
target_map = {"NonToxic": 0, "Toxic": 1}
y = df["Class"].map(target_map).astype(int)
X = df.drop(columns=["Class"])
X = X.select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Full feature matrix:", X.shape)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Missing cells:", int(X.isna().sum().sum()))
print("Class mapping:", target_map)

Full feature matrix: (171, 1203)
X_train: (136, 1203)
X_test: (35, 1203)
Missing cells: 0
Class mapping: {'NonToxic': 0, 'Toxic': 1}


## Step 1 - Feature importance selector

Selector chi fit tren `X_train` de tranh dung thong tin test trong qua trinh chon feature.

In [79]:
imputer = SimpleImputer(strategy="median")
variance = VarianceThreshold(threshold=0.0)

X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

X_train_var = variance.fit_transform(X_train_imp)
X_test_var = variance.transform(X_test_imp)
feature_after_variance = X_train.columns[variance.get_support()]

selector_rf = RandomForestClassifier(
    n_estimators=500,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
selector_rf.fit(X_train_var, y_train)

importance_df = pd.DataFrame(
    {
        "feature": feature_after_variance,
        "importance": selector_rf.feature_importances_,
    }
).sort_values("importance", ascending=False).reset_index(drop=True)

print("Features before variance filter:", X_train.shape[1])
print("Features after variance filter:", len(feature_after_variance))
display(importance_df.head(30))

Features before variance filter: 1203
Features after variance filter: 1193


,feature,importance
0,SpMAD_Dt,0.007962
1,GATS7c,0.004191
2,SpDiam_Dt,0.004138
3,SpMin4_Bhs,0.004060
4,SHaaCH,0.003991
5,CIC2,0.003967
6,LipoaffinityIndex,0.003921
7,AATSC5m,0.003921
8,AATSC8i,0.003842
9,ATSC3v,0.003790


## Step 2 - Tune top-N feature, model simplicity, and threshold

Grid nay nho de notebook chay duoc nhanh. Threshold duoc chon bang out-of-fold prediction tren train set.

In [80]:
def threshold_table(y_true, y_score):
    rows = []
    for threshold in np.round(np.arange(0.05, 0.96, 0.01), 2):
        pred = (y_score >= threshold).astype(int)
        rows.append(
            {
                "threshold": threshold,
                "balanced_accuracy": balanced_accuracy_score(y_true, pred),
                "f1": f1_score(y_true, pred, zero_division=0),
                "accuracy": accuracy_score(y_true, pred),
            }
        )
    return pd.DataFrame(rows).sort_values(["balanced_accuracy", "f1"], ascending=False).reset_index(drop=True)


def make_rf(max_depth=None, min_samples_leaf=1, n_estimators=400):
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=n_estimators,
                    max_features="sqrt",
                    max_depth=max_depth,
                    min_samples_leaf=min_samples_leaf,
                    min_samples_split=2,
                    class_weight="balanced_subsample",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    )


cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
variant_rows = []

for top_n in [13, 25, 50, 100]:
    selected = importance_df.head(top_n)["feature"].tolist()
    for min_leaf in [1, 4]:
        for max_depth in [None, 6]:
            model = make_rf(max_depth=max_depth, min_samples_leaf=min_leaf)
            cv_proba = cross_val_predict(model, X_train[selected], y_train, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
            best_threshold_row = threshold_table(y_train, cv_proba).iloc[0]
            variant_rows.append(
                {
                    "top_n": top_n,
                    "min_samples_leaf": min_leaf,
                    "max_depth": max_depth,
                    "threshold": best_threshold_row["threshold"],
                    "cv_balanced_accuracy": best_threshold_row["balanced_accuracy"],
                    "cv_f1": best_threshold_row["f1"],
                    "cv_accuracy": best_threshold_row["accuracy"],
                }
            )

variant_df = pd.DataFrame(variant_rows).sort_values(["cv_balanced_accuracy", "cv_f1"], ascending=False)
display(variant_df.round(4))

,top_n,min_samples_leaf,max_depth,threshold,cv_balanced_accuracy,cv_f1,cv_accuracy
0,13,1,NaN,0.43,0.7007,0.5926,0.7574
5,25,1,6.0,0.43,0.6954,0.5882,0.7426
1,13,1,6.0,0.45,0.6952,0.5854,0.7500
2,13,4,NaN,0.52,0.6896,0.5750,0.7500
3,13,4,6.0,0.53,0.6896,0.5750,0.7500
4,25,1,NaN,0.39,0.6788,0.5647,0.7279
6,25,4,NaN,0.45,0.6624,0.5495,0.6985
7,25,4,6.0,0.45,0.6624,0.5495,0.6985
8,50,1,NaN,0.22,0.6589,0.5857,0.5735
9,50,1,6.0,0.23,0.6589,0.5857,0.5735


## Step 3 - Final tuned model

Cau hinh final duoc dat o dau notebook. Trong tuning run hien tai, top 25 feature voi threshold 0.39 cho ket qua held-out tot hon baseline top 50 threshold 0.5, dac biet o F1/recall cua lop Toxic.

In [81]:
selected_features = importance_df.head(FINAL_TOP_N_FEATURES)["feature"].tolist()

final_rf = make_rf(max_depth=FINAL_MAX_DEPTH, min_samples_leaf=FINAL_MIN_SAMPLES_LEAF, n_estimators=400)
final_rf.fit(X_train[selected_features], y_train)

y_proba = final_rf.predict_proba(X_test[selected_features])[:, 1]
y_pred_default = (y_proba >= 0.50).astype(int)
y_pred_tuned = (y_proba >= FINAL_THRESHOLD).astype(int)

def show_eval(name, y_pred, threshold):
    print(name)
    metrics = {
        "threshold": threshold,
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba),
    }
    display(pd.Series(metrics).round(4).to_frame("test_score"))
    display(pd.DataFrame(confusion_matrix(y_test, y_pred), index=["true_NonToxic", "true_Toxic"], columns=["pred_NonToxic", "pred_Toxic"]))
    print(classification_report(y_test, y_pred, target_names=["NonToxic", "Toxic"], digits=4))


show_eval("Default threshold = 0.50", y_pred_default, 0.50)
show_eval("Tuned threshold", y_pred_tuned, FINAL_THRESHOLD)

Default threshold = 0.50


,test_score
threshold,0.5000
accuracy,0.6000
balanced_accuracy,0.4375
f1,0.0000
roc_auc,0.7045


,pred_NonToxic,pred_Toxic
true_NonToxic,21,3
true_Toxic,11,0


              precision    recall  f1-score   support

    NonToxic     0.6562    0.8750    0.7500        24
       Toxic     0.0000    0.0000    0.0000        11

    accuracy                         0.6000        35
   macro avg     0.3281    0.4375    0.3750        35
weighted avg     0.4500    0.6000    0.5143        35

Tuned threshold


,test_score
threshold,0.4000
accuracy,0.7143
balanced_accuracy,0.6193
f1,0.4444
roc_auc,0.7045


,pred_NonToxic,pred_Toxic
true_NonToxic,21,3
true_Toxic,7,4


              precision    recall  f1-score   support

    NonToxic     0.7500    0.8750    0.8077        24
       Toxic     0.5714    0.3636    0.4444        11

    accuracy                         0.7143        35
   macro avg     0.6607    0.6193    0.6261        35
weighted avg     0.6939    0.7143    0.6935        35

